In [92]:
import os
import time
import warnings
import json
import pmlb
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import train_test_split
import gc
import flaml
from flaml.automl.spark.utils import build_test_spark
warnings.filterwarnings("ignore")
spark = build_test_spark()
os.environ["FLAML_MAX_CONCURRENT"] = "2"
    

In [93]:
try:
    result = json.load(open("choice_result.json", "r"))
except:
    result = {}
BENCHMARK_TASKS = {}
for clas_task in pmlb.classification_dataset_names:
    BENCHMARK_TASKS[clas_task] = {
        "dataset": clas_task,
        "task": "classification",
        "metric": "accuracy",
        "size": "m",
    }
for reg_task in pmlb.regression_dataset_names:
    BENCHMARK_TASKS[reg_task] = {
        "dataset": reg_task,
        "task": "regression",
        "metric": "r2",
        "size": "m",
    }
ITERS_SIZE = {
    "xl": 4,
    "l": 8,
    "m": 12,
    "s": 20,
}

In [94]:
BENCHMARK_TASKS

{'GAMETES_Epistasis_2_Way_1000atts_0.4H_EDM_1_EDM_1_1': {'dataset': 'GAMETES_Epistasis_2_Way_1000atts_0.4H_EDM_1_EDM_1_1',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'm'},
 'GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1': {'dataset': 'GAMETES_Epistasis_2_Way_20atts_0.1H_EDM_1_1',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'm'},
 'GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1': {'dataset': 'GAMETES_Epistasis_2_Way_20atts_0.4H_EDM_1_1',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'm'},
 'GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1': {'dataset': 'GAMETES_Epistasis_3_Way_20atts_0.2H_EDM_1_1',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'm'},
 'GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_50_EDM_2_001': {'dataset': 'GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_50_EDM_2_001',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'm'},
 'GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_75_EDM_2_001': {'dataset': 'GAM

In [95]:
df_summary = pmlb.dataset_lists.df_summary
df_summary

,dataset,n_instances,n_features,n_binary_features,n_categorical_features,n_continuous_features,endpoint_type,n_classes,imbalance,task
0,1027_ESL,488,4,0,0,4,continuous,9.0,0.099363,regression
1,1028_SWD,1000,10,0,0,10,continuous,4.0,0.108291,regression
2,1029_LEV,1000,4,0,0,4,continuous,5.0,0.111245,regression
3,1030_ERA,1000,4,0,0,4,continuous,9.0,0.031251,regression
4,1089_USCrime,47,13,0,0,13,continuous,42.0,0.002970,regression
...,...,...,...,...,...,...,...,...,...,...
279,wine_quality_red,1599,11,0,0,11,categorical,6.0,0.228804,classification
280,wine_quality_white,4898,11,0,0,11,categorical,7.0,0.211974,classification
281,wine_recognition,178,13,0,2,11,categorical,3.0,0.012530,classification
282,xd6,973,9,9,0,0,categorical,2.0,0.114332,classification


In [96]:
def runner_flaml(task, use_spark=False):
    conf = BENCHMARK_TASKS[task]
    try:
        X, y = pmlb.fetch_data(conf['dataset'], return_X_y=True)
        print(f"Running on dataset: {conf['dataset']}")
    except Exception as e:
        print(f"{conf['dataset']} is not in pmlb database")
        return {"score":0, "duration":0}
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

    aml = flaml.AutoML()

    automl_settings = {
        "max_iter": ITERS_SIZE.get(conf["size"], 4),
        "metric": conf["metric"],
        "task": conf["task"],
        "estimator_list": ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth'],
    }
    
    if use_spark:
        automl_settings["use_spark"] = True
        automl_settings["n_concurrent_trials"] = 2
    
    if conf["task"] == "classification":
        automl_settings["estimator_list"] += ["lrl1"]
        
    start = time.time()
    aml.fit(X_train=X_train, y_train=y_train, **automl_settings)
    end = time.time()
    
    pred = aml.predict(X_test)
    
    if conf["task"] == "classification":
        score = accuracy_score(y_test, pred)
    else:
        score = r2_score(y_test, pred)
    del X, y, X_train, y_train, X_test, y_test, pred, aml
    return {
        "score": score,
        "duration": end - start,
        "metric": conf["metric"],
    }


In [97]:
for task in BENCHMARK_TASKS.keys():
    if task not in result.keys():
        result[task] = runner_flaml(task)
        json.dump(result, open("result.json", "w"))
        gc.collect()

In [98]:
selected_tasks = []
for task in result.keys():
    metric = result[task]["metric"]
    score = result[task]["score"]
    # Retain datasets where FLAML performed well, yet still possess room for improvement
    if 0.9 >= score >= 0.6:
        print(f"Task {task} score of {metric}: {score}")
        selected_tasks.append({"dataset":task,"metric":metric, "result":score})

Task GAMETES_Heterogeneity_20atts_1600_Het_0.4_0.2_75_EDM_2_001 score of accuracy: 0.74
Task adult score of accuracy: 0.8656129719105724
Task analcatdata_aids score of accuracy: 0.6923076923076923
Task analcatdata_asbestos score of accuracy: 0.8095238095238095
Task analcatdata_bankruptcy score of accuracy: 0.7692307692307693
Task analcatdata_boxing1 score of accuracy: 0.7333333333333333
Task analcatdata_boxing2 score of accuracy: 0.8181818181818182
Task analcatdata_cyyoung8092 score of accuracy: 0.8
Task analcatdata_cyyoung9302 score of accuracy: 0.8695652173913043
Task analcatdata_fraud score of accuracy: 0.7272727272727273
Task analcatdata_japansolvent score of accuracy: 0.6923076923076923
Task appendicitis score of accuracy: 0.8888888888888888
Task australian score of accuracy: 0.8728323699421965
Task auto score of accuracy: 0.7843137254901961
Task backache score of accuracy: 0.7777777777777778
Task balance_scale score of accuracy: 0.8789808917197452
Task breast_cancer score of accu

In [99]:
datasets_n_dict = dict(zip(df_summary["dataset"].values.tolist(), df_summary["n_instances"].values.tolist()))

In [100]:
for i in range(len(selected_tasks)):
   selected_tasks[i]["n"] = datasets_n_dict[selected_tasks[i]["dataset"]]
selected_tasks.sort(key=lambda x: x["n"])

selected_cla_tasks = [task for task in selected_tasks if task["metric"] == "accuracy"]
selected_reg_tasks = [task for task in selected_tasks if task["metric"] == "r2"]

In [104]:
import numpy as np
select_n = 7
def choice_task(tasks):
    selected_i = np.geomspace(1, len(tasks), select_n).astype('int')
    selected_i = len(tasks) - selected_i
    choice = []
    for ii in range(len(selected_i)):
        i = selected_i[ii]
        new_task = {}
        new_task["dataset"] = tasks[i]["dataset"]
        new_task["task"] = "regression" if tasks[i]["metric"] == "r2" else "classification"
        new_task["metric"] = tasks[i]["metric"]
        if i == len(tasks) - 1:
            size = "xl"
        elif i == 0:
            size = "s"
        elif ii <= len(selected_i) / 2:
            size = "l"
        else:
            size = "m"
        new_task["size"] = size
        choice.append(new_task)
    
    return choice

choice_cla_tasks = choice_task(selected_cla_tasks)
choice_reg_tasks = choice_task(selected_reg_tasks)
final_choice_tasks = choice_reg_tasks + choice_cla_tasks

In [105]:
final_choice_tasks

[{'dataset': '1203_BNG_pwLinear',
  'task': 'regression',
  'metric': 'r2',
  'size': 'xl'},
 {'dataset': '1193_BNG_lowbwt',
  'task': 'regression',
  'metric': 'r2',
  'size': 'l'},
 {'dataset': '537_houses', 'task': 'regression', 'metric': 'r2', 'size': 'l'},
 {'dataset': '294_satellite_image',
  'task': 'regression',
  'metric': 'r2',
  'size': 'l'},
 {'dataset': '598_fri_c0_1000_25',
  'task': 'regression',
  'metric': 'r2',
  'size': 'm'},
 {'dataset': '627_fri_c2_500_10',
  'task': 'regression',
  'metric': 'r2',
  'size': 'm'},
 {'dataset': '1089_USCrime',
  'task': 'regression',
  'metric': 'r2',
  'size': 's'},
 {'dataset': 'sleep',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'xl'},
 {'dataset': 'fars',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'l'},
 {'dataset': 'adult',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'l'},
 {'dataset': 'satimage',
  'task': 'classification',
  'metric': 'accuracy',
  'size': 'l'},
 {'datase

In [106]:
json.dump(final_choice_tasks, open("choice_tasks.json", "w"))